# Aula 10 — Relatório + Defesa

Esta é a aula final do Intensivão. Não vamos construir mais código — vamos **usar tudo que foi construído** para montar a entrega e aprender a defende-la.

A diferença entre uma equipe que perde e uma que ganha muitas vezes não está na estratégia em si. Está em **como ela é apresentada, justificada e defendida**.

---

**Agenda:**
1. Os critérios de avaliação — o que o júri pontua
2. Mapeando o nosso trabalho para cada critério
3. Estrutura do relatório final
4. Como defender escolhas metodológicas
5. Sessão de cross-review entre equipes
6. Checklist final de entrega

In [ ]:
# ── CONFIGURAÇÃO DO AMBIENTE ─────────────────────────────────────────────────
# Este notebook roda no VS Code (Jupyter local) E no Google Colab.
# Execute esta célula primeiro — ela detecta o ambiente e configura os caminhos.

import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ── Google Colab ──────────────────────────────────────────────────────────
    print("Ambiente detectado: Google Colab")

    # Montar Google Drive para persistir os dados entre sessões
    from google.colab import drive
    drive.mount('/content/drive')

    # Instalar dependências não incluídas no Colab por padrão
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'yfinance', 'pyarrow'], check=False)

    # Clonar o repositório do curso (pula automaticamente se já existir)
    REPO_DIR = '/content/intensivao-quant-ai'
    if not os.path.exists(REPO_DIR):
        print("Clonando repositório do curso...")
        subprocess.run([
            'git', 'clone',
            'https://github.com/fmaldonadoo/intensivao-quant-ai.git',
            REPO_DIR
        ])
        print("Repositório clonado com sucesso.")

    # Pasta de dados no Google Drive — persiste entre sessões do Colab
    DADOS_DIR = '/content/drive/MyDrive/intensivao_quant/dados'
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")

else:
    # ── VS Code / Jupyter local ───────────────────────────────────────────────
    print("Ambiente detectado: VS Code / Jupyter local")

    # Sobe a árvore de diretórios até encontrar a raiz do repositório (.git)
    _p = os.path.abspath(os.getcwd())
    while _p != os.path.dirname(_p):
        if os.path.exists(os.path.join(_p, '.git')):
            break
        _p = os.path.dirname(_p)

    DADOS_DIR = os.path.join(_p, 'dados')
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Carregando toda a cadeia de dados produzida no intensivão
ret_mensais = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'))
ret_ibov    = ret_mensais['BOVA11.SA']
ret_v1      = pd.read_parquet(os.path.join(DADOS_DIR, 'retorno_carteira_v1.parquet')).squeeze()
ret_v2      = pd.read_parquet(os.path.join(DADOS_DIR, 'retorno_carteira_sinal_v2.parquet')).squeeze()
ret_wf      = pd.read_parquet(os.path.join(DADOS_DIR, 'retorno_walkforward_liquido.parquet')).squeeze()
pesos_v2    = pd.read_parquet(os.path.join(DADOS_DIR, 'pesos_sinal_v2.parquet'))
sinal_v2    = pd.read_parquet(os.path.join(DADOS_DIR, 'sinal_v2.parquet'))

def calcular_metricas(retornos, benchmark=None, nome='Estratégia', rf_mensal=0.0):
    r = retornos.dropna()
    excesso = r - rf_mensal
    n = len(r)
    cagr = (1 + r).prod() ** (12 / n) - 1
    vol  = r.std() * np.sqrt(12)
    sharpe = (excesso.mean() / excesso.std()) * np.sqrt(12) if excesso.std() > 0 else np.nan
    downside = excesso[excesso < 0].std() * np.sqrt(12)
    sortino  = (excesso.mean() * 12 / downside) if downside > 0 else np.nan
    cum = (1 + r).cumprod()
    dd  = cum / cum.cummax() - 1
    mdd = dd.min()
    calmar = cagr / abs(mdd) if mdd != 0 else np.nan
    alpha, beta = np.nan, np.nan
    if benchmark is not None:
        b = benchmark.reindex(r.index).dropna()
        r2 = r.reindex(b.index)
        if len(r2) > 10:
            beta, intercept, *_ = stats.linregress(b.values, r2.values)
            alpha = intercept * 12
    return dict(nome=nome, cagr=cagr, vol=vol, sharpe=sharpe, sortino=sortino,
                max_drawdown=mdd, calmar=calmar, alpha=alpha, beta=beta,
                n_meses=n, inicio=r.index[0], fim=r.index[-1])

m_v1   = calcular_metricas(ret_v1,  benchmark=ret_ibov, nome='Sinal v1 (bruto)')
m_v2   = calcular_metricas(ret_v2,  benchmark=ret_ibov, nome='Sinal v2 (vol-adj.)')
m_wf   = calcular_metricas(ret_wf,  benchmark=ret_ibov, nome='Walk-forward líquido')
m_ibov = calcular_metricas(ret_ibov, nome='IBOVESPA')
turnover_medio = (pesos_v2.diff().abs().sum(axis=1) / 2).dropna().mean()

print('Dados carregados. Pronto para montar a entrega final.')

---
## 1. Os critérios de avaliação

O Desafio Itaú Asset Management avalia as equipes por múltiplos critérios. Antes de montar qualquer relatório, leia o edital completo e mapeie cada critério.

**Regra de ouro:** o relatório não é sobre o que você fez — é sobre o que o júri pergunta.

### Como pensar em critérios de competição quant

Tipicamente, competições desse tipo avaliam:

| Dimensão | O que o júri quer ver |
|----------|----------------------|
| **Tese** | Existe uma hipótese clara com fundamento econômico? |
| **Dados** | Os dados foram tratados com rigor? Vieses identificados? |
| **Metodologia** | As escolhas foram justificadas, não apenas otimizadas? |
| **Backtest** | Look-ahead? Custos? Walk-forward? |
| **Resultados** | As métricas são apresentadas honestamente? |
| **Análise de risco** | O que pode dar errado está documentado? |
| **Comunicação** | O relatório é claro para quem não conhece o código? |

**Atenção:** verifique o edital do seu ano — pesos e critérios mudam. Ajuste a ênfase do relatório conforme os pesos explícitos.

In [ ]:
# Diagrama de cobertura: o que cada aula entregou por dimensão avaliativa

criterios = [
    'Tese e fundamentação',
    'Qualidade dos dados',
    'Rigor metodológico',
    'Integridade do backtest',
    'Métricas e resultados',
    'Análise de risco',
    'Comunicação',
]

# Cobertura por aula (escala 0-1)
cobertura = {
    'Aula 1\n(Kickoff)':        [0.9, 0.0, 0.2, 0.2, 0.0, 0.1, 0.5],
    'Aula 2\n(Dados)':          [0.0, 0.9, 0.3, 0.4, 0.0, 0.2, 0.3],
    'Aula 3\n(EDA)':            [0.2, 1.0, 0.4, 0.3, 0.1, 0.5, 0.3],
    'Aula 4\n(Sinal v1)':       [0.8, 0.3, 0.7, 0.5, 0.4, 0.3, 0.4],
    'Aula 5\n(Backtest v1)':    [0.3, 0.2, 0.6, 0.7, 0.9, 0.5, 0.5],
    'Aula 6\n(Alocação)':       [0.5, 0.1, 0.8, 0.5, 0.7, 0.4, 0.5],
    'Aula 7\n(Sinal v2)':       [0.6, 0.2, 0.9, 0.7, 0.8, 0.5, 0.4],
    'Aula 8\n(Backtest rig.)':  [0.2, 0.3, 0.8, 1.0, 0.7, 0.9, 0.5],
    'Aula 9\n(GenAI)':          [0.3, 0.1, 0.4, 0.3, 0.5, 0.4, 1.0],
    'Aula 10\n(Defesa)':        [0.7, 0.2, 0.6, 0.5, 0.5, 0.7, 1.0],
}

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(criterios))
n_aulas = len(cobertura)
width = 0.08
colors = plt.cm.Blues(np.linspace(0.3, 0.9, n_aulas))

for i, (nome_aula, vals) in enumerate(cobertura.items()):
    offset = (i - n_aulas / 2) * width + width / 2
    bars = ax.bar(x + offset, vals, width * 0.9, label=nome_aula,
                  color=colors[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(criterios, fontsize=9)
ax.set_ylabel('Cobertura (0 → 1)')
ax.set_title('Mapeamento do intensivão para os critérios de avaliação')
ax.legend(loc='upper right', fontsize=7, ncol=2)
ax.set_ylim(0, 1.15)

plt.tight_layout()
plt.savefig('mapeamento_criterios.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 2. O argumento central — a "linha vermelha" do relatório

Todo relatório forte tem uma **linha vermelha**: um argumento central que conecta cada seção.

Para nossa estratégia, a linha vermelha é:

> *"Momentum cross-sectional no IBOVESPA, quando construído com fundamento econômico sólido, rigor metodológico e validação out-of-sample líquida de custos, gera alpha consistente — demonstrando que padrões comportamentais de underreaction persistem no mercado brasileiro."*

Cada seção do relatório deve **fortalecer** ou **qualificar honestamente** esse argumento:

```
Tese          → Por que momentum existe (Jegadeesh & Titman, 1993; underreaction)
Dados         → Que mercado estamos observando e quão limpos são os dados
Metodologia   → Como construímos o sinal — e por que essa escolha
Resultados    → Os números suportam a tese?
Risco         → Em que condições a tese falha?
Conclusão     → O argumento se sustenta, com as devidas qualificações
```

---
## 3. Estrutura do relatório final

A seguir, o template recomendado com indicação de tamanho e artefatos esperados em cada seção.

In [ ]:
# Template do relatório final
template_relatorio = [
    {
        'secao': 'Capa',
        'tamanho': '1 pág.',
        'conteudo': 'Título, equipe, instituição, data, resumo em 3 linhas',
        'artefatos': '—',
        'aula_origem': '—',
    },
    {
        'secao': '1. Resumo Executivo',
        'tamanho': '1 pág.',
        'conteudo': 'Tese, resultado-chave, método, limitação principal',
        'artefatos': 'Tabela de métricas simplificada',
        'aula_origem': 'Aula 9 (GenAI)',
    },
    {
        'secao': '2. Tese de Investimento',
        'tamanho': '1-2 pág.',
        'conteudo': 'Por que momentum funciona, evidências empíricas, mercado brasileiro',
        'artefatos': 'Gráfico de probabilidade condicional (Aula 4)',
        'aula_origem': 'Aulas 1 e 4',
    },
    {
        'secao': '3. Dados e Universo',
        'tamanho': '1 pág.',
        'conteudo': 'Fonte, período, filtros, NaNs, ajuste de dividendos',
        'artefatos': 'Tabela de cobertura por ticker, histograma de retornos',
        'aula_origem': 'Aulas 2 e 3',
    },
    {
        'secao': '4. Metodologia',
        'tamanho': '2-3 pág.',
        'conteudo': 'Sinal (fórmula), normalização, ranking, alocação, rebalanceamento',
        'artefatos': 'Linha do tempo do sinal (diagrama 12-1), equações em LaTeX',
        'aula_origem': 'Aulas 4, 6 e 7',
    },
    {
        'secao': '5. Resultados',
        'tamanho': '2-3 pág.',
        'conteudo': 'Todas as métricas (IS, WF, líquidas), tear sheet, heatmap',
        'artefatos': 'tearsheet_v1.png, heatmap mensal, tabela comparativa v1/v2/WF',
        'aula_origem': 'Aulas 5, 7 e 8',
    },
    {
        'secao': '6. Análise de Risco',
        'tamanho': '1-2 pág.',
        'conteudo': 'Survivorship bias, custos (break-even), regimes adversos a momentum',
        'artefatos': 'Gráfico break-even de custos, rolling Sharpe',
        'aula_origem': 'Aulas 3 e 8',
    },
    {
        'secao': '7. Conclusão',
        'tamanho': '0.5 pág.',
        'conteudo': 'Reafirma tese, reconhece limitações, propõe extensões',
        'artefatos': '—',
        'aula_origem': 'Aula 9 (GenAI)',
    },
    {
        'secao': '8. Referências',
        'tamanho': '0.5 pág.',
        'conteudo': 'J&T 1993, DeMiguel 2009, Bailey & López de Prado 2014',
        'artefatos': '—',
        'aula_origem': '—',
    },
]

df_template = pd.DataFrame(template_relatorio).set_index('secao')
print('=== TEMPLATE DO RELATÓRIO FINAL ===')
print(df_template[['tamanho', 'aula_origem']].to_string())
print(f'\nTotal estimado: ~10-12 páginas (excluindo capa e referências)')

---
## 4. O tear sheet final — a imagem mais importante do relatório

Gestores profissionais são acostumados a tear sheets. Uma boa tear sheet comunica em segundos o que a estratégia faz. Vamos produzir a versão final, combinando todos os resultados do intensivão.

In [ ]:
# Tear sheet final — comparação completa de todas as versões
fig = plt.figure(figsize=(16, 12))
fig.suptitle('Estratégia de Momentum Cross-Sectional — IBOVESPA\nTear Sheet Final',
             fontsize=14, fontweight='bold', y=0.98)

gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.35)

# --- Painel 1: Retorno Cumulativo ---
ax1 = fig.add_subplot(gs[0, :])
idx_comum = ret_v1.dropna().index.intersection(ret_v2.dropna().index).intersection(ret_wf.dropna().index)

((1 + ret_v1.reindex(idx_comum)).cumprod() - 1).plot(
    ax=ax1, color='steelblue', lw=1.5, alpha=0.7, label='Sinal v1 (bruto, IS)')
((1 + ret_v2.reindex(idx_comum)).cumprod() - 1).plot(
    ax=ax1, color='royalblue', lw=2, label='Sinal v2 (vol-adj., IS)')
((1 + ret_wf.reindex(idx_comum)).cumprod() - 1).plot(
    ax=ax1, color='navy', lw=2.5, linestyle='--', label='Walk-forward líquido (OOS)')
((1 + ret_ibov.reindex(idx_comum)).cumprod() - 1).plot(
    ax=ax1, color='gray', lw=1.5, linestyle=':', label='IBOVESPA')
ax1.set_title('Retorno Cumulativo')
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xlabel('')

# --- Painel 2: Drawdown (walk-forward) ---
ax2 = fig.add_subplot(gs[1, :2])
cum_wf = (1 + ret_wf.reindex(idx_comum)).cumprod()
dd_wf  = cum_wf / cum_wf.cummax() - 1
ax2.fill_between(dd_wf.index, dd_wf.values, 0, color='tomato', alpha=0.6)
ax2.plot(dd_wf.index, dd_wf.values, color='darkred', lw=0.8)
ax2.set_title('Drawdown — Walk-forward líquido')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax2.set_xlabel('')

# --- Painel 3: Rolling Sharpe 24m ---
ax3 = fig.add_subplot(gs[1, 2])
rolling_sh = ret_wf.reindex(idx_comum).rolling(24).apply(
    lambda r: (r.mean()/r.std())*np.sqrt(12) if r.std() > 0 else np.nan)
rolling_sh.plot(ax=ax3, color='navy', lw=1.5)
ax3.axhline(0, color='black', lw=0.8)
ax3.axhline(1.0, color='green', lw=1, linestyle='--', alpha=0.7)
ax3.set_title('Rolling Sharpe 24m (OOS)')
ax3.set_xlabel('')

# --- Painel 4: Retorno anual ---
ax4 = fig.add_subplot(gs[2, :2])
ret_anual = ret_wf.reindex(idx_comum).resample('YE').apply(lambda r: (1+r).prod()-1)
ibov_anual = ret_ibov.reindex(idx_comum).resample('YE').apply(lambda r: (1+r).prod()-1)
anos = ret_anual.index.year
x = np.arange(len(anos))
ax4.bar(x - 0.2, ret_anual.values, 0.38,
        color=['steelblue' if v >= 0 else 'tomato' for v in ret_anual.values],
        label='Estratégia WF', alpha=0.85)
ax4.bar(x + 0.2, ibov_anual.values, 0.38,
        color=['lightgray' if v >= 0 else 'salmon' for v in ibov_anual.values],
        label='IBOVESPA', alpha=0.7)
ax4.set_xticks(x)
ax4.set_xticklabels(anos, rotation=45, fontsize=8)
ax4.axhline(0, color='black', lw=0.8)
ax4.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax4.set_title('Retorno Anual — WF vs IBOVESPA')
ax4.legend(fontsize=8)

# --- Painel 5: Tabela de métricas ---
ax5 = fig.add_subplot(gs[2, 2])
ax5.axis('off')

metricas_tabela = [
    ['Métrica', 'Estratégia', 'IBOVESPA'],
    ['CAGR', f"{m_wf['cagr']:.1%}", f"{m_ibov['cagr']:.1%}"],
    ['Vol. Anual', f"{m_wf['vol']:.1%}", f"{m_ibov['vol']:.1%}"],
    ['Sharpe', f"{m_wf['sharpe']:.2f}", f"{m_ibov['sharpe']:.2f}"],
    ['Sortino', f"{m_wf['sortino']:.2f}", f"{m_ibov['sortino']:.2f}"],
    ['Max DD', f"{m_wf['max_drawdown']:.1%}", f"{m_ibov['max_drawdown']:.1%}"],
    ['Alpha a.a.', f"{m_wf['alpha']:.1%}", '—'],
    ['Beta', f"{m_wf['beta']:.2f}", '—'],
    ['Turnover/mês', f"{turnover_medio:.0%}", '—'],
]
tabela = ax5.table(cellText=metricas_tabela[1:], colLabels=metricas_tabela[0],
                   loc='center', cellLoc='center')
tabela.auto_set_font_size(False)
tabela.set_fontsize(8)
tabela.scale(1.0, 1.4)
for (row, col), cell in tabela.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c4a7c')
        cell.set_text_props(color='white', fontweight='bold')
    elif col == 1 and row > 0:
        cell.set_facecolor('#e8f0fe')
ax5.set_title('Walk-forward líquido\n(custo 0.5%/turno)', fontsize=8, pad=12)

plt.savefig('tearsheet_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tear sheet final salvo: tearsheet_final.png')

---
## 5. Defendendo escolhas metodológicas

### A estrutura de uma boa defesa

Para qualquer escolha metodológica, a defesa tem três camadas:

```
1. FUNDAMENTO      → Existe evidência teórica ou empírica para essa escolha?
2. TRADE-OFFS      → Quais alternativas foram consideradas e por que foram preteridas?
3. SENSIBILIDADE   → A estratégia funciona se esse parâmetro variar razoavelmente?
```

### Exemplos de perguntas e respostas

Vamos praticar as perguntas mais comuns do júri.

In [ ]:
# Compilando as questões mais comuns e estruturando respostas

defesa = [
    {
        'pergunta': 'Por que a janela 12-1 meses?',
        'fundamento': 'Jegadeesh & Titman (1993) demonstraram empiricamente que retornos entre '
                      '3 e 12 meses atrás têm poder preditivo positivo. O skip do mês mais recente '
                      'evita o efeito de reversão de curto prazo documentado na literatura.',
        'alternativas': '3-1, 6-1, 24-1 foram exploradas; 12-1 tem a maior base de evidências '
                        'internacionais e resultados consistentes na amostra brasileira.',
        'sensibilidade': 'Janelas entre 9 e 15 meses produzem resultados qualitativamente '
                         'similares, confirmando robustez.',
    },
    {
        'pergunta': 'Por que vol de 63 dias para normalizar?',
        'fundamento': 'Queremos capturar volatilidade recente sem ruído excessivo. '
                      '63 dias ≈ 1 trimestre — horizonte natural para revisão de risco em '
                      'gestão profissional. Shorter (21d) é ruidoso; longer (252d) reage tarde.',
        'alternativas': 'Testamos 21d, 63d e 252d. 63d melhor equilibra reatividade e estabilidade '
                        '(menor variância do Sharpe rolling).',
        'sensibilidade': 'Resultados são robustos para janelas entre 42 e 126 dias.',
    },
    {
        'pergunta': 'Por que top 10% e não top 20% ou top 5 ações?',
        'fundamento': 'Top 10% equilibra concentração de sinal (quanto maior o threshold, '
                      'mais puro o sinal) e diversificação (quanto menor, mais concentrado e '
                      'dependente de idiossincrasias individuais).',
        'alternativas': 'Top 5% é muito concentrado (3-4 ações) para um backtest mensal confiável. '
                        'Top 20% dilui o sinal com ações medianas no ranking.',
        'sensibilidade': 'Top 8-12% produz métricas semelhantes; a estratégia não é \'fragile\' '
                         'a essa escolha.',
    },
    {
        'pergunta': 'Por que equal weight e não signal-weighted?',
        'fundamento': 'DeMiguel et al. (2009) mostraram que equal weight supera portfólios '
                      'otimizados em dados reais em 14 das 14 métricas testadas, devido a '
                      'erros de estimação da covariância.',
        'alternativas': 'Testamos equal weight, signal-weighted e inverse-vol na Aula 6. '
                        'Equal weight obteve Sharpe mais consistente e menor turnover.',
        'sensibilidade': 'Signal-weighted entregou alpha similar com maior turnover (custo maior).',
    },
    {
        'pergunta': 'Os resultados são reais ou overfitting?',
        'fundamento': 'A janela 12-1 foi fixada antes de ver os dados, baseada em literatura '
                      'de 1993. Isso minimiza o número efetivo de testes para ~1.',
        'alternativas': 'Validação walk-forward (treino 48m, teste 12m) produz métricas '
                        'out-of-sample consistentes com in-sample. Degradação de '
                        f'{(m_wf["sharpe"]/m_v2["sharpe"]-1):.0%} é compatível com literatura.',
        'sensibilidade': 'Deflated Sharpe com ~1 teste efetivo: nossa estratégia é '
                         'estatisticamente significativa.',
    },
]

for item in defesa:
    print(f"\n{'='*70}")
    print(f"P: {item['pergunta']}")
    print(f"\n  Fundamento:")
    print(f"    {item['fundamento']}")
    print(f"\n  Alternativas consideradas:")
    print(f"    {item['alternativas']}")
    print(f"\n  Sensibilidade:")
    print(f"    {item['sensibilidade']}")

---
## 6. Falhas que contam como pontos

Um erro comum de equipes de primeira viagem: esconder limitações. O efeito é o oposto do esperado — o júri percebe, e a credibilidade da equipe despenca.

**Limitações bem documentadas demonstram maturidade metodológica.**

### Como reportar uma limitação corretamente

```
FRACO: "Nossa estratégia não usa survivorship bias."
         ↑ Vago, não quantifica, parece desculpa.

FORTE: "Utilizamos o constituinte atual do IBOVESPA como universo fixo, o que
         introduz survivorship bias positivo — estimamos que isso pode inflar o
         CAGR em 1-3% a.a. com base na literatura (Elton et al., 1996). Uma
         extensão natural é usar os constituintes históricos point-in-time."
         ↑ Quantifica, cita, propõe solução. Isso é maturidade.
```

In [ ]:
# Tabela de limitações — o que incluir no relatório

limitacoes = pd.DataFrame([
    {
        'Limitação': 'Survivorship bias',
        'Impacto estimado': '+1-3% CAGR a.a.',
        'Direção': 'Infla resultados',
        'Mitigação adotada': 'Filtro de liquidez p20 reduz ações marginais',
        'Extensão proposta': 'Usar constituintes históricos (Economatica/Bloomberg)',
    },
    {
        'Limitação': 'Dados não point-in-time',
        'Impacto estimado': 'Desconhecido',
        'Direção': 'Potencialmente infla',
        'Mitigação adotada': 'Preços de fechamento ajustados — impacto reduzido para momentum puro',
        'Extensão proposta': 'Provider com dados P.I.T. (ex: Refinitiv)',
    },
    {
        'Limitação': 'Custos modelados (não reais)',
        'Impacto estimado': '±0.5 Sharpe dependendo do custo real',
        'Direção': 'Ambígua',
        'Mitigação adotada': 'Análise de break-even: estratégia positiva até ~X% por turno',
        'Extensão proposta': 'Paper trading com custo real observado',
    },
    {
        'Limitação': 'Concentração (7-8 ações)',
        'Impacto estimado': 'Risco idiossincrático elevado',
        'Direção': 'Risco bilateral',
        'Mitigação adotada': 'Equal weight diversifica dentro do portfólio',
        'Extensão proposta': 'Ampliar para top 15-20% ou adicionar restrição de setor',
    },
    {
        'Limitação': 'Escopo único de fator',
        'Impacto estimado': 'Vulnerável a regimes anti-momentum',
        'Direção': 'Risco de crise (momentum crash)',
        'Mitigação adotada': 'Volatility adjustment (sinal v2) mitiga parcialmente',
        'Extensão proposta': 'Combinar momentum com value ou quality (multi-fator)',
    },
])

print('=== LIMITAÇÕES A INCLUIR NO RELATÓRIO ===')
print(limitacoes[['Limitação', 'Impacto estimado', 'Direção', 'Extensão proposta']].to_string(index=False))

---
## 7. Sessão de cross-review entre equipes

O cross-review é a parte final do intensivão. Cada equipe atua como **júri** das outras.

### Por que fazer isso

1. Aprender com variações metodológicas das outras equipes
2. Praticar a defesa sob pressão antes do dia real
3. Identificar pontos cegos que você não veria sozinho
4. Desenvolver a capacidade de avaliar estratégias quantitativas

### Protocolo da sessão

In [ ]:
# Rubrica de avaliação para cross-review

rubrica = pd.DataFrame([
    {'Critério': 'Clareza da tese',
     'Pontos': 10,
     'Pergunta-guia': 'Em uma frase, qual a hipótese central e está bem fundamentada?'},
    {'Critério': 'Rigor dos dados',
     'Pontos': 15,
     'Pergunta-guia': 'Os filtros foram aplicados? Vieses identificados e quantificados?'},
    {'Critério': 'Justificativa dos parâmetros',
     'Pontos': 20,
     'Pergunta-guia': 'Cada escolha tem fundamento econômico ou foi simplesmente otimizada?'},
    {'Critério': 'Integridade do backtest',
     'Pontos': 20,
     'Pergunta-guia': 'Há look-ahead? Walk-forward? Custos modelados?'},
    {'Critério': 'Qualidade dos resultados',
     'Pontos': 15,
     'Pergunta-guia': 'As métricas são plausíveis? Sharpe muito alto (>3) levanta suspeitas.'},
    {'Critério': 'Análise de risco',
     'Pontos': 10,
     'Pergunta-guia': 'As limitações foram reconhecidas honestamente?'},
    {'Critério': 'Comunicação e clareza',
     'Pontos': 10,
     'Pergunta-guia': 'Alguém sem ver o código entende a estratégia?'},
])

print(f'=== RUBRICA DO CROSS-REVIEW ===')
print(rubrica.to_string(index=False))
print(f'\nTotal: {rubrica["Pontos"].sum()} pontos')

In [ ]:
# Visualização da rubrica em gráfico de rosca
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico de pizza — distribuição dos pontos
ax = axes[0]
cores = ['#1f4e79', '#2e75b6', '#4472c4', '#5b9bd5', '#70ad47', '#ffc000', '#ff0000']
wedges, texts, autotexts = ax.pie(
    rubrica['Pontos'],
    labels=[f"{r['Critério']}\n({r['Pontos']}pts)" for _, r in rubrica.iterrows()],
    colors=cores,
    autopct='%1.0f%%',
    startangle=90,
    pctdistance=0.75,
)
for text in texts:
    text.set_fontsize(8)
ax.set_title('Distribuição de pontos\nna rubrica de cross-review', fontweight='bold')

# Timeline da sessão de cross-review
ax = axes[1]
ax.axis('off')

etapas = [
    ('00:00', 'Apresentação (10 min)', '#1f4e79'),
    ('00:10', 'Perguntas do júri (15 min)', '#c00000'),
    ('00:25', 'Deliberação interna (5 min)', '#7f7f7f'),
    ('00:30', 'Feedback estruturado (10 min)', '#375623'),
    ('00:40', 'Troca de papéis: próxima equipe', '#4472c4'),
]

for i, (tempo, descricao, cor) in enumerate(etapas):
    y = 0.85 - i * 0.18
    ax.add_patch(mpatches.FancyBboxPatch((0.05, y - 0.06), 0.9, 0.12,
                                          boxstyle='round,pad=0.02', facecolor=cor, alpha=0.15,
                                          edgecolor=cor, linewidth=2))
    ax.text(0.12, y, tempo, ha='left', va='center', fontsize=10,
            fontweight='bold', color=cor)
    ax.text(0.30, y, descricao, ha='left', va='center', fontsize=10)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('Timeline da sessão de cross-review\n(por equipe avaliada)', fontweight='bold')

plt.tight_layout()
plt.savefig('crossreview_rubrica.png', dpi=120, bbox_inches='tight')
plt.show()

### As 5 perguntas que o júri sempre faz

Durante o cross-review, concentre as perguntas nas áreas com maior sinal-ruído:

1. **"Como vocês escolheram o parâmetro X?"**  
   → Procure por: "testamos vários e ficamos com o melhor" (red flag de overfitting)

2. **"Qual é a explicação econômica para essa estratégia funcionar?"**  
   → Procure por: ausência de hipótese comportamental ou estrutural

3. **"O que acontece com a estratégia em março de 2020?"** (COVID crash)  
   → Momentum sofre em reversões abruptas. Eles sabem disso?

4. **"Qual é o Sharpe out-of-sample?"**  
   → Se só reportam in-sample, pergunta crítica.

5. **"Se eu aumentar o custo de transação para 1%, a estratégia ainda funciona?"**  
   → Testa robustez a premissas de custo.

---
## 8. Checklist final de entrega

In [ ]:
checklist_final = [
    # Código e dados
    ('CÓDIGO', 'Notebooks numerados e funcionando do início ao fim sem erros'),
    ('CÓDIGO', 'Parâmetros principais em variáveis nomeadas (sem magic numbers)'),
    ('CÓDIGO', 'Todos os parquets gerados e presentes na pasta /dados'),
    ('CÓDIGO', 'Nenhum look-ahead bias: shift(2) no sinal, shift(1) nos pesos'),
    ('CÓDIGO', 'requirements.txt ou environment.yml com versões fixadas'),
    # Metodologia
    ('MÉTODO', 'Janela de momentum justificada com referência (J&T 1993)'),
    ('MÉTODO', 'Volatilidade rolling: janela escolhida com fundamento documentado'),
    ('MÉTODO', 'Alocação: por que equal weight (DeMiguel 2009 citado)'),
    ('MÉTODO', 'Turnover calculado e reportado'),
    # Backtest
    ('BACKTEST', 'Walk-forward: resultado out-of-sample presente'),
    ('BACKTEST', 'Custos de transação modelados (premissa explícita)'),
    ('BACKTEST', 'Análise de break-even de custos incluída'),
    ('BACKTEST', 'Survivorship bias reconhecido e quantificado aproximadamente'),
    # Relatório
    ('RELATÓRIO', 'Resumo executivo: 1 página com tese + resultado principal + limitação'),
    ('RELATÓRIO', 'Todas as métricas: CAGR, vol, Sharpe, Sortino, MDD, alpha, beta'),
    ('RELATÓRIO', 'Tear sheet incluída como figura'),
    ('RELATÓRIO', 'Seção de análise de risco com limitações documentadas'),
    ('RELATÓRIO', 'Referências: J&T 1993, DeMiguel 2009, Bailey & LdP 2014'),
    # Apresentação
    ('DEFESA', 'Cada escolha metodológica: fundamento → alternativas → sensibilidade'),
    ('DEFESA', 'Respostas para: "por que essa janela?", "por que equal weight?"'),
    ('DEFESA', 'Simulado de cross-review realizado com outra equipe'),
    ('DEFESA', 'Slides têm no máximo 15 palavras por slide'),
]

df_checklist = pd.DataFrame(checklist_final, columns=['Categoria', 'Item'])

print('=== CHECKLIST FINAL DE ENTREGA ===')
cat_atual = ''
for _, row in df_checklist.iterrows():
    if row['Categoria'] != cat_atual:
        cat_atual = row['Categoria']
        print(f"\n{'─'*50}")
        print(f" {cat_atual}")
        print(f"{'─'*50}")
    print(f"  [ ] {row['Item']}")

print(f"\nTotal de itens: {len(df_checklist)}")

---
## 9. O que separa os finalistas

Ao avaliar centenas de equipes em competições desse tipo, gestores experientes relatam que os finalistas têm em comum:

### Critérios de differenciação

| Aspecto | Maioria das equipes | Finalistas |
|---------|---------------------|------------|
| **Parâmetros** | "Testamos vários e ficamos com o melhor" | "Fixamos pela literatura antes de ver os dados" |
| **Limitações** | Ignoradas ou minimizadas | Quantificadas e com extensões propostas |
| **Custos** | Ignorados | Modelados com análise de sensibilidade |
| **Validação** | In-sample apenas | Walk-forward OOS explícito |
| **Sharpe alto** | Comemoram sem questionar | Investigam se há look-ahead ou overfitting |
| **Defesa** | Defensiva quando questionados | Antecipam críticas no relatório |
| **Comunicação** | Código no slide | Gráficos que contam uma história |

### A frase que resume tudo

> *"Não precisamos de uma estratégia perfeita. Precisamos de uma estratégia honesta, fundamentada, e bem comunicada — e da capacidade de defender cada escolha com evidência."*

In [ ]:
# Visão geral do intensivão: o que construímos

print('=== INTENSIVÃO QUANT AI — RESUMO DO QUE CONSTRUÍMOS ===')
print()
entregas = [
    ('Aula 1', 'Kickoff', 'Alinhamento sobre o que é uma estratégia quant e os critérios de avaliação'),
    ('Aula 2', 'Dados', 'Pipeline de coleta: ~77 tickers IBOV, retornos log, parquets'),
    ('Aula 3', 'EDA', 'Estatísticas descritivas, fat tails, ADF, ACF, filtros de qualidade'),
    ('Aula 4', 'Sinal v1', 'Momentum 12-1, ranking cross-sectional, turnover, primeira carteira'),
    ('Aula 5', 'Backtest v1', 'CAGR, Sharpe, Sortino, drawdown, Calmar, alpha/beta, tear sheet'),
    ('Aula 6', 'Alocação', 'Fronteira eficiente, equal vs signal vs inv-vol, DeMiguel 2009'),
    ('Aula 7', 'Sinal v2', 'Normalização por vol realizada 63d, comparação v1 vs v2'),
    ('Aula 8', 'Backtest rig.', 'Look-ahead, overfitting, multiple testing, DSR, custos, walk-forward'),
    ('Aula 9', 'GenAI', 'API Claude: resumo executivo, metodologia, análise de risco, draft'),
    ('Aula 10', 'Defesa', 'Tear sheet final, rubrica de cross-review, checklist de entrega'),
]
for aula, nome, entrega in entregas:
    print(f'  {aula:7s} — {nome:<18s} → {entrega}')

print()
print('Arquivos gerados:')
print('  dados/precos_ibov.parquet')
print('  dados/retornos_diarios_limpo.parquet')
print('  dados/retornos_mensais_limpo.parquet')
print('  dados/sinal_v1.parquet, sinal_v2.parquet')
print('  dados/pesos_v1.parquet, pesos_sinal_v2.parquet')
print('  dados/retorno_carteira_v1.parquet, ..._v2.parquet, ..._walkforward_liquido.parquet')
print('  aula-05-backtest-v1/tearsheet_v1.png')
print('  aula-09-genai-analise/relatorio_draft.md')
print('  aula-10-relatorio-defesa/tearsheet_final.png')

---
## Fim do Intensivão Quant AI

Em 10 aulas, construímos do zero:

- Uma estratégia quantitativa com fundamento econômico sólido
- Um pipeline de dados replicável e documentado
- Um backtest rigoroso com walk-forward e custos modelados
- Três esquemas de alocação comparados com evidência
- Um relatório draft gerado com IA e revisado criticamente
- Uma estrutura de defesa para o dia da apresentação

O trabalho de vocês nas equipes agora é:

1. Replicar o pipeline em equipe, entendendo cada linha
2. Documentar as decisões da equipe que divergirem das do intensivão
3. Fazer o cross-review com pelo menos uma outra equipe antes de entregar
4. Ler o edital linha a linha e mapear cada critério para o relatório

---

*Boa sorte no Desafio Itaú Asset Management. O mais importante que esse intensivão quis transmitir: um resultado honesto e bem defendido vale mais do que um Sharpe impressionante sem substância.*